# AiPO projekt — sprawdzenie inferencji modelu

Ten notebook uruchamia sam model YOLO na nagraniu i zapisuje tabelę detekcji do CSV. Jest dobry do szybkiej kontroli, czy model wykrywa klasy tak, jak oczekujesz, zanim odpalisz cały pipeline z trackingiem i statystykami.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


## Ustaw ścieżki

Wrzuć nagranie do `content/`, a model YOLO do `models/`.


In [ ]:
VIDEO_PATH = PROJECT_ROOT / "content" / "example_match.mp4"
MODEL_PATH = PROJECT_ROOT / "models" / "best.pt"
OUTPUT_CSV = PROJECT_ROOT / "outputs" / "model_detections.csv"

CONFIDENCE = 0.30
MAX_FRAMES = 200


In [ ]:
assert VIDEO_PATH.exists(), f"Nie znaleziono nagrania: {VIDEO_PATH}"
assert MODEL_PATH.exists(), f"Nie znaleziono modelu: {MODEL_PATH}"


## Uruchom inferencję


In [ ]:
from aipo_football_cv.infer_model import infer_frames

df = infer_frames(
    model_path=MODEL_PATH,
    video_path=VIDEO_PATH,
    output_csv=OUTPUT_CSV,
    confidence=CONFIDENCE,
    max_frames=MAX_FRAMES,
)

df.head()


## Podsumowanie klas


In [ ]:
summary = (
    df.groupby("class_name")
    .agg(count=("class_name", "size"), avg_confidence=("confidence", "mean"))
    .sort_values("count", ascending=False)
)
summary


## Prosty wykres liczby detekcji po klasach


In [ ]:
import matplotlib.pyplot as plt

summary["count"].plot(kind="bar")
plt.title("Liczba detekcji według klas")
plt.xlabel("Klasa")
plt.ylabel("Liczba detekcji")
plt.show()
